# 04 — Linear Regression From Scratch (Sections 10, 11, 12)

Covers synthetic regression dataset design, manual/from-scratch linear regression training pipeline, and controlled optimization experiments.

**Compliance (from-scratch boundary):** no `sklearn.linear_model.LinearRegression`, no `torch.optim.*`, no `torch.nn.Module`/`nn.Linear`/`nn.Parameter`, no `nn` losses/modules, no `torch.nn.functional` loss/objective helpers, no `torch.utils.data` training abstractions. Uses raw tensors with `requires_grad=True`, manual forward/loss/gradient descent updates in `torch.no_grad()`, and explicit gradient clearing.


In [1]:
import os
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

os.makedirs('../reports', exist_ok=True)
os.makedirs('../experiments', exist_ok=True)
print('Artifacts dirs ready: reports/, experiments/')


Artifacts dirs ready: reports/, experiments/


## Section 10 — Linear Model, Data, and Manual Components
### 1. Generate the dataset (1 input feature, 1 continuous target, known coefficient/intercept, added noise)
### 2. Inspect the relationship between inputs and targets
### 3. Split the dataset into training and validation sets
### 4. Define the model's weight and bias as trainable parameters
### 5. Define the prediction operation
### 6. Define the MSE loss
### 7. Design the gradient descent update process


In [2]:
# Known data-generating process: y = 3.5*x + 1.2 + noise
true_w = 3.5
true_b = 1.2
n_samples = 120
noise_std = 0.5

x = torch.linspace(-2.0, 2.0, steps=n_samples).unsqueeze(1)
noise = torch.randn(n_samples, 1) * noise_std
y = true_w * x + true_b + noise

print('x shape:', tuple(x.shape), 'y shape:', tuple(y.shape))
print('Known parameters: w={}, b={}, noise_std={}'.format(true_w, true_b, noise_std))
print('x range:', x.min().item(), x.max().item(), 'y range:', y.min().item(), y.max().item())

plt.figure(figsize=(5,4))
plt.scatter(x.numpy(), y.numpy(), s=12, alpha=0.7, label='samples')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic linear dataset')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/section10_linear_data.png', dpi=120)
plt.show()

# Train/validation split
n_train = int(0.8 * n_samples)
indices = torch.randperm(n_samples)
train_idx, val_idx = indices[:n_train], indices[n_train:]
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]
print('Train size:', x_train.shape[0], 'Val size:', x_val.shape[0])

# Trainable parameters (raw tensors)
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

def predict(x_in, w_in, b_in):
  '''Linear prediction.'''
  return x_in * w_in + b_in

def mse_loss(y_hat, y_true):
  '''Mean squared error.'''
  return ((y_hat - y_true) ** 2).mean()

def gd_step(params, lr):
  '''Manual gradient descent update step.'''
  with torch.no_grad():
    for p in params:
      if p.grad is not None:
        p -= lr * p.grad

print('Initial params: w=', w.item(), 'b=', b.item())
print('Prediction op, MSE loss, and GD update process defined.')


x shape: (120, 1) y shape: (120, 1)
Known parameters: w=3.5, b=1.2, noise_std=0.5
x range: -2.0 2.0 y range: -6.36292028427124 8.322158813476562


Train size: 96 Val size: 24
Initial params: w= -1.7946094274520874 b= -0.2341044843196869
Prediction op, MSE loss, and GD update process defined.


/tmp/ipykernel_69026/4193125783.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 11 — Complete Training Pipeline
### Implement complete linear regression training pipeline (manual, from-scratch)


In [3]:
def train_linear_regression(x_tr, y_tr, x_va, y_va, lr=0.05, epochs=200, init_w=None, init_b=None, seed=0):
  '''Train linear regression from scratch and return history/results.'''
  torch.manual_seed(seed)
  w_p = torch.randn(1) if init_w is None else torch.tensor([float(init_w)], dtype=torch.float32)
  b_p = torch.randn(1) if init_b is None else torch.tensor([float(init_b)], dtype=torch.float32)
  w_p = w_p.clone().detach().requires_grad_(True)
  b_p = b_p.clone().detach().requires_grad_(True)
  train_losses = []
  val_losses = []
  for epoch in range(epochs):
    # forward
    y_hat = predict(x_tr, w_p, b_p)
    loss = mse_loss(y_hat, y_tr)
    # backward
    loss.backward()
    # update
    gd_step([w_p, b_p], lr)
    # clear gradients
    w_p.grad.zero_()
    b_p.grad.zero_()
    # track
    train_losses.append(loss.item())
    with torch.no_grad():
      val_pred = predict(x_va, w_p, b_p)
      val_loss = mse_loss(val_pred, y_va)
      val_losses.append(val_loss.item())
  return {
    'w': w_p.detach().item(),
    'b': b_p.detach().item(),
    'train_losses': train_losses,
    'val_losses': val_losses,
  }

result = train_linear_regression(x_train, y_train, x_val, y_val, lr=0.05, epochs=300, seed=42)
print('Learned w:', result['w'], 'Known w:', true_w)
print('Learned b:', result['b'], 'Known b:', true_b)
print('Final train loss:', result['train_losses'][-1])
print('Final val loss:', result['val_losses'][-1])

plt.figure(figsize=(6,4))
plt.plot(result['train_losses'], label='train loss')
plt.plot(result['val_losses'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Section 11: Training and validation loss')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/section11_linear_training_loss.png', dpi=120)
plt.show()


Learned w: 3.51435923576355 Known w: 3.5
Learned b: 1.2079209089279175 Known b: 1.2
Final train loss: 0.2627417743206024
Final val loss: 0.2893196642398834


/tmp/ipykernel_69026/4276378595.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 12 — Controlled Experiments (A-F)
Run experiments with different learning rates, initialization, noise, and sample sizes. For each: record config, train, record train/val loss, plot loss curve, compare learned parameters, explain results.


In [4]:
def run_experiment(name, lr, epochs, init_w, init_b, seed, n_total, noise, note):
  torch.manual_seed(seed)
  np.random.seed(seed)
  x_e = torch.linspace(-2.0, 2.0, steps=n_total).unsqueeze(1)
  y_e = true_w * x_e + true_b + torch.randn(n_total, 1) * noise
  idx = torch.randperm(n_total)
  n_tr = int(0.8 * n_total)
  x_tr, y_tr = x_e[idx[:n_tr]], y_e[idx[:n_tr]]
  x_va, y_va = x_e[idx[n_tr:]], y_e[idx[n_tr:]]
  res = train_linear_regression(x_tr, y_tr, x_va, y_va, lr=lr, epochs=epochs, init_w=init_w, init_b=init_b, seed=seed)
  res.update({'name': name, 'config': {'lr': lr, 'epochs': epochs, 'init_w': init_w, 'init_b': init_b, 'seed': seed, 'n_total': n_total, 'noise_std': noise}, 'note': note})
  return res

experiments = [
  run_experiment('A Very small learning rate', lr=0.001, epochs=400, init_w=0.0, init_b=0.0, seed=1, n_total=120, noise=0.5, note='Slow convergence expected'),
  run_experiment('B Reasonable learning rate', lr=0.05, epochs=300, init_w=0.0, init_b=0.0, seed=2, n_total=120, noise=0.5, note='Stable convergence expected'),
  run_experiment('C Excessively large learning rate', lr=1.2, epochs=80, init_w=0.0, init_b=0.0, seed=3, n_total=120, noise=0.5, note='Instability/divergence risk'),
  run_experiment('D Different initialization', lr=0.05, epochs=300, init_w=-5.0, init_b=8.0, seed=4, n_total=120, noise=0.5, note='Sensitivity to initialization'),
  run_experiment('E Different dataset noise', lr=0.05, epochs=300, init_w=0.0, init_b=0.0, seed=5, n_total=120, noise=2.0, note='Higher noise impacts estimation/val loss'),
  run_experiment('F Different sample sizes', lr=0.05, epochs=300, init_w=0.0, init_b=0.0, seed=6, n_total=24, noise=0.5, note='Smaller sample affects parameter estimates'),
]

for i, exp in enumerate(experiments, 1):
  print('='*80)
  print(exp['name'])
  print('Config:', exp['config'])
  print('Final train loss:', exp['train_losses'][-1], 'Final val loss:', exp['val_losses'][-1])
  print('Learned w,b:', exp['w'], exp['b'], 'Known w,b:', true_w, true_b)
  print('Explanation:', exp['note'])
  plt.figure(figsize=(5,3.5))
  plt.plot(exp['train_losses'], label='train')
  plt.plot(exp['val_losses'], label='val')
  plt.title(exp['name'])
  plt.xlabel('Epoch')
  plt.ylabel('MSE')
  plt.legend()
  plt.tight_layout()
  path = '../reports/section12_exp_{:02d}_loss.png'.format(i)
  plt.savefig(path, dpi=120)
  plt.show()
  print('Saved loss curve:', path)

print('Completed experiments A-F.')


A Very small learning rate
Config: {'lr': 0.001, 'epochs': 400, 'init_w': 0.0, 'init_b': 0.0, 'seed': 1, 'n_total': 120, 'noise_std': 0.5}
Final train loss: 2.5034878253936768 Final val loss: 2.1130473613739014
Learned w,b: 2.3561060428619385 0.7161818742752075 Known w,b: 3.5 1.2
Explanation: Slow convergence expected
Saved loss curve: ../reports/section12_exp_01_loss.png
B Reasonable learning rate
Config: {'lr': 0.05, 'epochs': 300, 'init_w': 0.0, 'init_b': 0.0, 'seed': 2, 'n_total': 120, 'noise_std': 0.5}
Final train loss: 0.2653891146183014 Final val loss: 0.15136028826236725
Learned w,b: 3.548032522201538 1.2189202308654785 Known w,b: 3.5 1.2
Explanation: Stable convergence expected


/tmp/ipykernel_69026/3946250674.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved loss curve: ../reports/section12_exp_02_loss.png
C Excessively large learning rate
Config: {'lr': 1.2, 'epochs': 80, 'init_w': 0.0, 'init_b': 0.0, 'seed': 3, 'n_total': 120, 'noise_std': 0.5}
Final train loss: inf Final val loss: inf
Learned w,b: -3.556414953827767e+30 -1.0057708224473967e+30 Known w,b: 3.5 1.2
Explanation: Instability/divergence risk
Saved loss curve: ../reports/section12_exp_03_loss.png
D Different initialization
Config: {'lr': 0.05, 'epochs': 300, 'init_w': -5.0, 'init_b': 8.0, 'seed': 4, 'n_total': 120, 'noise_std': 0.5}
Final train loss: 0.25911185145378113 Final val loss: 0.3205774426460266
Learned w,b: 3.4671967029571533 1.162266731262207 Known w,b: 3.5 1.2
Explanation: Sensitivity to initialization


Saved loss curve: ../reports/section12_exp_04_loss.png
E Different dataset noise
Config: {'lr': 0.05, 'epochs': 300, 'init_w': 0.0, 'init_b': 0.0, 'seed': 5, 'n_total': 120, 'noise_std': 2.0}
Final train loss: 4.52638053894043 Final val loss: 4.3899455070495605
Learned w,b: 3.698225975036621 1.352047324180603 Known w,b: 3.5 1.2
Explanation: Higher noise impacts estimation/val loss


Saved loss curve: ../reports/section12_exp_05_loss.png
F Different sample sizes
Config: {'lr': 0.05, 'epochs': 300, 'init_w': 0.0, 'init_b': 0.0, 'seed': 6, 'n_total': 24, 'noise_std': 0.5}
Final train loss: 0.16712011396884918 Final val loss: 0.14714817702770233
Learned w,b: 3.5361626148223877 1.1495143175125122 Known w,b: 3.5 1.2
Explanation: Smaller sample affects parameter estimates
Saved loss curve: ../reports/section12_exp_06_loss.png
Completed experiments A-F.


## Summary
- Implemented synthetic linear regression data, manual trainable parameters, prediction, MSE, and GD update.
- Built complete training pipeline with epoch-wise train/val tracking and parameter comparison.
- Ran controlled experiments A-F, saved loss curves, and compared convergence/estimation behavior.
